# Map creation module

This module is made to create map representation of the data

In [6]:
# Necessary imports
import folium
import pandas as pd
import json

Map was downloaded from : https://github.com/gregoiredavid/france-geojson/blob/master/communes.geojson

In [7]:
# Load the GeoJSON map of France communes
with open("map/communes.geojson") as f:
    communes_geojson = json.load(f)

# Create a map centered on France
m = folium.Map(location=[46.603354, 1.888334], zoom_start=6)

Get the data : it is important that the column 'codecommune' (used for matching with the map) is kept alongside everydata that needs to be ploted on the map

## To modify -----------------------------------------------------

Mock example : map of the leading candidate in the 2022 presidental election (Macron - yellow vs Lepen - Blue)

In [8]:
# Load the election results data
election_data = pd.read_csv("data/elections/pres2022_csv/pres2022comm.csv", low_memory=False)  # replace with your file path

# Find the leading candidate for each commune and map to 'M' or 'L'
def get_leading_candidate(row):
    if row['voixT2MACRON'] >= row['voixT2MLEPEN']:
        return 'M'  # Macron
    else:
        return 'L'  # Le Pen

# Apply the function to get the leading candidate
election_data['leading_candidate'] = election_data.apply(get_leading_candidate, axis=1)

# Map commune codes to election results
election_dict = election_data[['codecommune', 'leading_candidate']]

Style function should match color to codecommune

In [9]:
def style_function(feature):
    # Get the commune code and result
    codecommune = feature["properties"]["code"]
    result = election_dict.loc[election_dict['codecommune'] == codecommune, 'leading_candidate']
    if not result.empty:
        leading_candidate = result.iloc[0] 
    else:
        leading_candidate = None 
    
    # Set color based on the leading candidate
    if leading_candidate == 'L':
        color = "#1f77b4"  # Blue for Le Pen
    elif leading_candidate == 'M':
        color = "#ffcc00"  # Yellow for Macron
    else:
        color = "#d9d9d9"  # Grey for no data

    return {"fillColor": color, "color": "black", "fillOpacity": 0.7, "weight": 0.3}

## -------------------------------------------------------

Will take some time : map creation

In [10]:
# Add GeoJSON layer with color styling
folium.GeoJson(
    communes_geojson,
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(fields=["code"]),
    name="Election Results by Commune"
).add_to(m)

# Add a layer control
folium.LayerControl().add_to(m)

# Save the map to an HTML file
m.save("map_created/france_communes_election_map.html")

A html file is created at the root of the project : open in a web browser to visualize it.